# Exploration SQL de la base DuckDB : fraude à l'ouverture de compte

Ce notebook interroge `data/fraud.duckdb` (table `applications`, construite par `src/build_duckdb.py`) uniquement en SQL :

1. **Contrôle qualité** : volumétrie, doublons, valeurs manquantes (vraies et codées), colonnes constantes, règles de validité, modalités, valeurs aberrantes
2. **Agrégations** : taux de fraude par mois, par modalité (`GROUPING SETS`), croisements (`PIVOT`, `ROLLUP`), profil fraudeurs / légitimes
3. **Fonctions de fenêtre** : évolution mensuelle (`LAG`, cumul, moyenne mobile), classement des modalités (`RANK`), déciles de risque (`NTILE`), variables calculées par ligne

La connexion est en **lecture seule** : les vues et tables créées ici sont temporaires (`TEMP`) et n'existent que pendant la session.

In [1]:
from pathlib import Path

import duckdb
import pandas as pd

ROOT = Path.cwd() if (Path.cwd() / "data").exists() else Path.cwd().parent
DB_PATH = ROOT / "data" / "fraud.duckdb"

con = duckdb.connect(str(DB_PATH), read_only=True)
pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 200)


def sql(query: str) -> pd.DataFrame:
    """Exécute une requête SQL sur la base et renvoie un DataFrame."""
    return con.sql(query).df()


sql("SHOW TABLES")

,name
0,applications


## 0. Aperçu de la table

In [2]:
sql("DESCRIBE applications")

,column_name,column_type,null,key,default,extra
0,fraud_bool,BIGINT,YES,None,None,None
1,income,DOUBLE,YES,None,None,None
2,name_email_similarity,DOUBLE,YES,None,None,None
3,prev_address_months_count,BIGINT,YES,None,None,None
4,current_address_months_count,BIGINT,YES,None,None,None
5,customer_age,BIGINT,YES,None,None,None
6,days_since_request,DOUBLE,YES,None,None,None
7,intended_balcon_amount,DOUBLE,YES,None,None,None
8,payment_type,VARCHAR,YES,None,None,None
9,zip_count_4w,BIGINT,YES,None,None,None


In [3]:
sql("SELECT * FROM applications LIMIT 5")

,fraud_bool,income,name_email_similarity,prev_address_months_count,current_address_months_count,customer_age,days_since_request,intended_balcon_amount,payment_type,zip_count_4w,velocity_6h,velocity_24h,velocity_4w,bank_branch_count_8w,date_of_birth_distinct_emails_4w,employment_status,credit_risk_score,email_is_free,housing_status,phone_home_valid,phone_mobile_valid,bank_months_count,has_other_cards,proposed_credit_limit,foreign_request,source,session_length_in_minutes,device_os,keep_alive_session,device_distinct_emails_8w,device_fraud_count,month,source_file
0,0,0.3,0.986506,-1,25,40,0.006735,102.453711,AA,1059,13096.035018,7850.955007,6742.080561,5,5,CB,163,1,BC,0,1,9,0,1500.0,0,INTERNET,16.224843,linux,1,1,0,0,dataset_part1.csv
1,0,0.8,0.617426,-1,89,20,0.010095,-0.849551,AD,1658,9223.283431,5745.251481,5941.664859,3,18,CA,154,1,BC,1,1,2,0,1500.0,0,INTERNET,3.363854,other,1,1,0,0,dataset_part1.csv
2,0,0.8,0.996707,9,14,40,0.012316,-1.490386,AB,1095,4471.472149,5471.988958,5992.555113,15,11,CA,89,1,BC,0,1,30,0,200.0,0,INTERNET,22.730559,windows,0,1,0,0,dataset_part1.csv
3,0,0.6,0.475100,11,14,30,0.006991,-1.863101,AB,3483,14431.993621,6755.344479,5970.336831,11,13,CA,90,1,BC,0,1,1,0,200.0,0,INTERNET,15.215816,linux,1,1,0,0,dataset_part1.csv
4,0,0.9,0.842307,-1,29,40,5.742626,47.152498,AA,2339,7601.511579,5124.046930,5940.734212,1,6,CA,91,0,BC,1,1,26,0,200.0,0,INTERNET,3.743048,other,0,1,0,0,dataset_part1.csv


`SUMMARIZE` donne en une requête min, max, moyenne, quartiles et pourcentage de `NULL` pour chaque colonne.

In [4]:
sql("""
    SELECT column_name, column_type, min, max, round(avg::DOUBLE, 3) AS moyenne,
           q25, q50, q75, approx_unique, null_percentage
    FROM (SUMMARIZE applications)
""")

,column_name,column_type,min,max,moyenne,q25,q50,q75,approx_unique,null_percentage
0,fraud_bool,BIGINT,0,1,0.011,0,0,0,2,0.0
1,income,DOUBLE,0.1,0.9,0.563,0.3,0.6000000000000001,0.8,9,0.0
2,name_email_similarity,DOUBLE,1.4345504845275636e-06,0.9999993177937188,0.494,0.2253460748039174,0.4915651662373566,0.7557359208826973,1167686,0.0
3,prev_address_months_count,BIGINT,-1,383,16.719,-1,-1,13,405,0.0
4,current_address_months_count,BIGINT,-1,428,86.588,19,52,130,468,0.0
5,customer_age,BIGINT,10,90,33.689,20,30,40,10,0.0
6,days_since_request,DOUBLE,4.036859788721786e-09,78.45690383509861,1.026,0.007204412123720937,0.015185471339041143,0.02634518655485249,888490,0.0
7,intended_balcon_amount,DOUBLE,-15.530554840076814,112.9569276953714,8.661,-1.1811450120380744,-0.8306819908900653,4.830282618960409,1004919,0.0
8,payment_type,VARCHAR,AA,AE,NaN,NaN,NaN,NaN,5,0.0
9,zip_count_4w,BIGINT,1,6700,1572.692,894,1263,1943,5921,0.0


## 1. Contrôle qualité

### 1.1 Volumétrie par fichier source
`ROLLUP` ajoute une ligne de total. On attend 1 000 000 lignes.

In [5]:
sql("""
    SELECT coalesce(source_file, 'TOTAL') AS source_file,
           count(*)                       AS lignes,
           min(month)                     AS mois_min,
           max(month)                     AS mois_max
    FROM applications
    GROUP BY ROLLUP (source_file)
    ORDER BY source_file = 'TOTAL', source_file
""")

,source_file,lignes,mois_min,mois_max
0,dataset_part1.csv,333334,0,2
1,dataset_part2.csv,333334,2,4
2,dataset_part3.csv,333332,4,7
3,TOTAL,1000000,0,7


Les fichiers ont été découpés par nombre de lignes et non par mois : les mois 2 et 4 sont répartis sur deux fichiers. `PIVOT` le montre directement.

In [6]:
sql("""
    PIVOT applications
    ON source_file
    USING count(*)
    GROUP BY month
    ORDER BY month
""")

,month,dataset_part1.csv,dataset_part2.csv,dataset_part3.csv
0,0,132440,0,0
1,1,127620,0,0
2,2,73274,63705,0
3,3,0,150936,0
4,4,0,118693,8998
5,5,0,0,119323
6,6,0,0,108168
7,7,0,0,96843


### 1.2 Doublons
Le jeu n'a pas d'identifiant : on compare le nombre de lignes au nombre de lignes distinctes (hors `source_file`).

In [7]:
sql("""
    WITH total AS (SELECT count(*) AS lignes FROM applications),
    distinctes AS (
        SELECT count(*) AS lignes_distinctes
        FROM (SELECT DISTINCT * EXCLUDE (source_file) FROM applications)
    )
    SELECT lignes, lignes_distinctes, lignes - lignes_distinctes AS doublons
    FROM total, distinctes
""")

,lignes,lignes_distinctes,doublons
0,1000000,1000000,0


### 1.3 Valeurs manquantes
Aucune colonne ne contient de vrai `NULL` (voir `null_percentage` plus haut). En revanche, la documentation Feedzai indique que les manquants sont **codés** :
`-1` pour 5 variables, et toute valeur négative pour `intended_balcon_amount`.

On repère d'abord toutes les colonnes numériques qui ont un minimum négatif :

In [8]:
sql("""
    SELECT column_name, min, max
    FROM (SUMMARIZE applications)
    WHERE column_type IN ('BIGINT', 'DOUBLE') AND TRY_CAST(min AS DOUBLE) < 0
    ORDER BY column_name
""")

,column_name,min,max
0,bank_months_count,-1,32
1,credit_risk_score,-170,389
2,current_address_months_count,-1,428
3,device_distinct_emails_8w,-1,2
4,intended_balcon_amount,-15.530554840076814,112.9569276953714
5,prev_address_months_count,-1,383
6,session_length_in_minutes,-1.0,85.89914319274027
7,velocity_6h,-170.60307235124628,16715.565404174275


Puis on compte les manquants codés en passant les colonnes en format long avec `UNPIVOT`.
`velocity_6h` a aussi quelques valeurs négatives, mais ce n'est pas un code de manquant documenté : on le traite comme une anomalie (section 1.5).

In [9]:
sql("""
    WITH long AS (
        UNPIVOT applications
        ON prev_address_months_count, current_address_months_count, bank_months_count,
           session_length_in_minutes, device_distinct_emails_8w, intended_balcon_amount
        INTO NAME variable VALUE valeur
    )
    SELECT variable,
           count(*) FILTER (WHERE CASE WHEN variable = 'intended_balcon_amount'
                                       THEN valeur < 0 ELSE valeur = -1 END)  AS manquants,
           round(100.0 * manquants / count(*), 2)                             AS pct_manquants,
           -- taux de fraude selon que la valeur est manquante ou non
           round(100 * avg(fraud_bool) FILTER (WHERE CASE WHEN variable = 'intended_balcon_amount'
                                                          THEN valeur < 0 ELSE valeur = -1 END), 2)
                                                                              AS taux_fraude_si_manquant,
           round(100 * avg(fraud_bool) FILTER (WHERE NOT CASE WHEN variable = 'intended_balcon_amount'
                                                              THEN valeur < 0 ELSE valeur = -1 END), 2)
                                                                              AS taux_fraude_si_renseigne
    FROM long
    GROUP BY variable
    ORDER BY pct_manquants DESC
""")

,variable,manquants,pct_manquants,taux_fraude_si_manquant,taux_fraude_si_renseigne
0,intended_balcon_amount,742523,74.25,1.31,0.50
1,prev_address_months_count,712920,71.29,1.42,0.31
2,bank_months_count,253635,25.36,1.63,0.92
3,current_address_months_count,4254,0.43,0.33,1.11
4,session_length_in_minutes,2015,0.20,0.89,1.10
5,device_distinct_emails_8w,359,0.04,1.11,1.10


Les effectifs correspondent à ceux de `lineage.json` (712 920, 742 523, 253 635…). Le fait qu'une valeur soit manquante est lui-même lié au taux de fraude : on gardera des indicateurs `*_manquant`.

### 1.4 Colonnes constantes
`count(DISTINCT COLUMNS(*))` compte les valeurs distinctes de toutes les colonnes en une seule passe.

In [10]:
distincts = sql("SELECT count(DISTINCT COLUMNS(*)) FROM applications").T
distincts.columns = ["valeurs_distinctes"]
distincts.sort_values("valeurs_distinctes").head(10)

,valeurs_distinctes
device_fraud_count,1
fraud_bool,2
keep_alive_session,2
source,2
foreign_request,2
has_other_cards,2
phone_mobile_valid,2
phone_home_valid,2
email_is_free,2
source_file,3


`device_fraud_count` ne prend qu'une seule valeur (0) : elle n'apporte aucune information et sera exclue.

### 1.5 Règles de validité
Chaque règle compte ses violations ; `UNPIVOT` remet le résultat en une ligne par règle.

In [11]:
sql("""
    UNPIVOT (
        SELECT
            count(*) FILTER (WHERE fraud_bool NOT IN (0, 1))                          AS "fraud_bool binaire",
            count(*) FILTER (WHERE least(email_is_free, phone_home_valid, phone_mobile_valid,
                                         has_other_cards, foreign_request, keep_alive_session) < 0
                                OR greatest(email_is_free, phone_home_valid, phone_mobile_valid,
                                            has_other_cards, foreign_request, keep_alive_session) > 1)
                                                                                      AS "indicateurs binaires 0/1",
            count(*) FILTER (WHERE round(income, 1) NOT BETWEEN 0.1 AND 0.9)          AS "income entre 0,1 et 0,9",
            count(*) FILTER (WHERE name_email_similarity NOT BETWEEN 0 AND 1)         AS "name_email_similarity entre 0 et 1",
            count(*) FILTER (WHERE customer_age % 10 <> 0
                                OR customer_age NOT BETWEEN 10 AND 90)                AS "customer_age tranche de 10 ans",
            count(*) FILTER (WHERE month NOT BETWEEN 0 AND 7)                         AS "month entre 0 et 7",
            count(*) FILTER (WHERE days_since_request < 0)                            AS "days_since_request >= 0",
            count(*) FILTER (WHERE velocity_6h < 0)                                   AS "velocity_6h >= 0",
            count(*) FILTER (WHERE velocity_24h < 0 OR velocity_4w < 0)               AS "velocity_24h et 4w >= 0",
            count(*) FILTER (WHERE zip_count_4w < 0 OR bank_branch_count_8w < 0
                                OR date_of_birth_distinct_emails_4w < 0)              AS "compteurs >= 0",
            count(*) FILTER (WHERE proposed_credit_limit <= 0)                        AS "proposed_credit_limit > 0",
            count(*) FILTER (WHERE source NOT IN ('INTERNET', 'TELEAPP'))             AS "source INTERNET/TELEAPP",
            count(*) FILTER (WHERE prev_address_months_count < -1
                                OR current_address_months_count < -1
                                OR bank_months_count < -1)                            AS "anciennetés >= -1"
        FROM applications
    )
    ON COLUMNS(*)
    INTO NAME regle VALUE violations
""")

,regle,violations
0,fraud_bool binaire,0
1,indicateurs binaires 0/1,0
2,"income entre 0,1 et 0,9",0
3,name_email_similarity entre 0 et 1,0
4,customer_age tranche de 10 ans,0
5,month entre 0 et 7,0
6,days_since_request >= 0,0
7,velocity_6h >= 0,44
8,velocity_24h et 4w >= 0,0
9,compteurs >= 0,0


Seule `velocity_6h` viole sa règle : une vitesse de demandes ne peut pas être négative. On regarde l'ampleur du problème :

In [12]:
sql("""
    SELECT count(*)                      AS lignes,
           min(velocity_6h)              AS min,
           max(velocity_6h)              AS max,
           round(100 * avg(fraud_bool), 2) AS taux_fraude_pct
    FROM applications
    WHERE velocity_6h < 0
""")

,lignes,min,max,taux_fraude_pct
0,44,-170.603072,-1.889614,0.0


### 1.6 Modalités des variables qualitatives

In [13]:
sql("""
    WITH long AS (
        UNPIVOT (SELECT payment_type, employment_status, housing_status, source, device_os
                 FROM applications)
        ON COLUMNS(*) INTO NAME variable VALUE modalite
    ),
    comptes AS (
        SELECT variable, modalite, count(*) AS n
        FROM long
        GROUP BY ALL
    )
    SELECT variable,
           count(*)                                                      AS nb_modalites,
           string_agg(modalite || ' (' || n || ')', ', ' ORDER BY n DESC) AS modalites_effectifs
    FROM comptes
    GROUP BY variable
    ORDER BY variable
""")

,variable,nb_modalites,modalites_effectifs
0,device_os,5,"other (342728), linux (332712), windows (26350..."
1,employment_status,7,"CA (730252), CB (138288), CF (44034), CC (3775..."
2,housing_status,7,"BC (372143), BB (260965), BA (169675), BE (169..."
3,payment_type,5,"AB (370554), AA (258249), AC (252071), AD (118..."
4,source,2,"INTERNET (992952), TELEAPP (7048)"


### 1.7 Vue nettoyée
On applique les corrections identifiées dans une vue temporaire `applications_clean` :

- manquants codés → `NULL`, avec un indicateur `<col>_manquant`
- `velocity_6h` négative → `NULL`
- suppression de `device_fraud_count` (constante)
- ajout d'un identifiant `application_id` et du découpage temporel `split` (train = mois 0 à 5, test = mois 6 et 7, comme dans `lineage.json`)

In [14]:
con.sql("""
    CREATE OR REPLACE TEMP VIEW applications_clean AS
    SELECT
        row_number() OVER ()                                         AS application_id,
        * EXCLUDE (prev_address_months_count, current_address_months_count, bank_months_count,
                   session_length_in_minutes, device_distinct_emails_8w, intended_balcon_amount,
                   velocity_6h, device_fraud_count),
        nullif(prev_address_months_count, -1)                        AS prev_address_months_count,
        nullif(current_address_months_count, -1)                     AS current_address_months_count,
        nullif(bank_months_count, -1)                                AS bank_months_count,
        nullif(session_length_in_minutes, -1)                        AS session_length_in_minutes,
        nullif(device_distinct_emails_8w, -1)                        AS device_distinct_emails_8w,
        CASE WHEN intended_balcon_amount >= 0 THEN intended_balcon_amount END AS intended_balcon_amount,
        CASE WHEN velocity_6h >= 0 THEN velocity_6h END              AS velocity_6h,
        (prev_address_months_count = -1)::INT                        AS prev_address_months_count_manquant,
        (current_address_months_count = -1)::INT                     AS current_address_months_count_manquant,
        (bank_months_count = -1)::INT                                AS bank_months_count_manquant,
        (session_length_in_minutes = -1)::INT                        AS session_length_in_minutes_manquant,
        (device_distinct_emails_8w = -1)::INT                        AS device_distinct_emails_8w_manquant,
        (intended_balcon_amount < 0)::INT                            AS intended_balcon_amount_manquant,
        CASE WHEN month <= 5 THEN 'train' ELSE 'test' END            AS split
    FROM applications
""")

sql("""
    SELECT count(*) AS lignes,
           count(*) FILTER (WHERE prev_address_months_count IS NULL) AS prev_address_null,
           count(*) FILTER (WHERE intended_balcon_amount IS NULL)    AS intended_balcon_null,
           count(*) FILTER (WHERE velocity_6h IS NULL)               AS velocity_6h_null
    FROM applications_clean
""")

,lignes,prev_address_null,intended_balcon_null,velocity_6h_null
0,1000000,712920,742523,44


### 1.8 Valeurs aberrantes (règle de l'IQR)
Une valeur est aberrante si elle sort de [Q1 − 1,5·IQR ; Q3 + 1,5·IQR]. Calculé sur la vue nettoyée, pour ne pas compter les `-1` comme extrêmes.

In [15]:
sql("""
    WITH long AS (
        UNPIVOT (SELECT velocity_6h, velocity_24h, velocity_4w, zip_count_4w, bank_branch_count_8w,
                        days_since_request, session_length_in_minutes, proposed_credit_limit,
                        intended_balcon_amount, credit_risk_score::DOUBLE AS credit_risk_score
                 FROM applications_clean)
        ON COLUMNS(*) INTO NAME variable VALUE valeur
    ),
    bornes AS (
        SELECT variable,
               quantile_cont(valeur, 0.25) AS q1,
               quantile_cont(valeur, 0.75) AS q3,
               q3 - q1                     AS iqr
        FROM long
        GROUP BY variable
    )
    SELECT variable,
           round(any_value(q1), 2)                                         AS q1,
           round(any_value(q3), 2)                                         AS q3,
           round(any_value(q3) + 1.5 * any_value(iqr), 2)                  AS borne_haute,
           round(max(valeur), 2)                                           AS max,
           count(*) FILTER (WHERE valeur < q1 - 1.5 * iqr
                               OR valeur > q3 + 1.5 * iqr)                 AS aberrants,
           round(100.0 * aberrants / count(*), 2)                          AS pct_aberrants
    FROM long JOIN bornes USING (variable)
    GROUP BY variable
    ORDER BY pct_aberrants DESC
""")

,variable,q1,q3,borne_haute,max,aberrants,pct_aberrants
0,proposed_credit_limit,200.00,500.00,950.00,2100.00,241742,24.17
1,bank_branch_count_8w,1.00,25.00,61.00,2385.00,175243,17.52
2,days_since_request,0.01,0.03,0.06,78.46,94834,9.48
3,session_length_in_minutes,3.12,8.88,17.52,85.90,78715,7.89
4,intended_balcon_amount,20.40,49.59,93.36,112.96,18369,7.13
5,zip_count_4w,894.00,1944.00,3519.00,6700.00,59871,5.99
6,velocity_6h,3436.66,7680.80,14047.02,16715.57,9007,0.90
7,credit_risk_score,83.00,178.00,320.50,389.00,8729,0.87
8,velocity_24h,3593.18,5752.57,8991.67,9506.90,2917,0.29
9,velocity_4w,4268.37,5488.08,7317.66,6994.76,0,0.00


À lire avec prudence :
- `proposed_credit_limit` est discrète (12 valeurs) : la règle de l'IQR n'a pas de sens ici, les « aberrants » sont simplement les limites hautes (1 000 et plus).
- `bank_branch_count_8w` et `days_since_request` sont très asymétriques : la plupart des valeurs sont proches de 0 et la queue est longue, ce qui reste plausible (agences très fréquentées, demandes anciennes).

On signale ces valeurs sans les supprimer.

## 2. Agrégations

### 2.1 Taux de fraude global et par échantillon

In [16]:
sql("""
    SELECT coalesce(split, 'TOTAL')         AS split,
           count(*)                         AS demandes,
           sum(fraud_bool)::BIGINT                  AS fraudes,
           round(100 * avg(fraud_bool), 3)  AS taux_fraude_pct
    FROM applications_clean
    GROUP BY ROLLUP (split)
    ORDER BY split = 'TOTAL', split DESC
""")

,split,demandes,fraudes,taux_fraude_pct
0,train,794989,8151,1.025
1,test,205011,2878,1.404
2,TOTAL,1000000,11029,1.103


### 2.2 Taux de fraude par modalité (`GROUPING SETS`)
Une seule requête calcule les agrégats pour les 5 variables qualitatives ; `GROUPING()` indique quelle variable est agrégée sur chaque ligne.

In [17]:
par_modalite = """
    SELECT
        CASE 0
            WHEN GROUPING(payment_type)      THEN 'payment_type'
            WHEN GROUPING(employment_status) THEN 'employment_status'
            WHEN GROUPING(housing_status)    THEN 'housing_status'
            WHEN GROUPING(source)            THEN 'source'
            WHEN GROUPING(device_os)         THEN 'device_os'
        END                                                                  AS variable,
        coalesce(payment_type, employment_status, housing_status, source, device_os) AS modalite,
        count(*)                                                             AS demandes,
        sum(fraud_bool)::BIGINT                                                      AS fraudes,
        round(100 * avg(fraud_bool), 3)                                      AS taux_fraude_pct
    FROM applications_clean
    GROUP BY GROUPING SETS ((payment_type), (employment_status), (housing_status), (source), (device_os))
"""
sql(par_modalite + " ORDER BY variable, taux_fraude_pct DESC")

,variable,modalite,demandes,fraudes,taux_fraude_pct
0,device_os,windows,263506,6507,2.469
1,device_os,macintosh,53826,752,1.397
2,device_os,x11,7228,81,1.121
3,device_os,other,342728,1974,0.576
4,device_os,linux,332712,1715,0.515
5,employment_status,CC,37758,932,2.468
6,employment_status,CG,453,7,1.545
7,employment_status,CA,730252,8899,1.219
8,employment_status,CB,138288,953,0.689
9,employment_status,CD,26522,100,0.377


### 2.3 Croisement âge × revenu (`PIVOT`)
Taux de fraude (%) par tranche d'âge (lignes) et décile de revenu (colonnes).

In [18]:
sql("""
    PIVOT (SELECT customer_age, round(income, 1)::VARCHAR AS income, fraud_bool FROM applications_clean)
    ON income
    USING round(100 * avg(fraud_bool), 2)
    GROUP BY customer_age
    ORDER BY customer_age
""")

,customer_age,0.1,0.2,0.3,0.4,0.5,0.6,0.7,0.8,0.9
0,10,0.22,0.19,0.44,0.46,0.43,0.43,0.46,0.45,0.46
1,20,0.29,0.30,0.27,0.37,0.48,0.40,0.47,0.55,1.06
2,30,0.48,0.54,0.56,0.60,0.50,0.65,0.66,0.82,1.59
3,40,0.64,0.80,0.65,0.69,0.91,0.95,0.85,1.11,2.10
4,50,1.17,1.06,1.33,1.27,1.54,1.64,1.56,1.79,3.45
5,60,1.80,2.01,2.51,2.70,2.11,2.59,2.82,3.32,5.92
6,70,2.31,2.13,1.95,3.51,1.97,3.87,3.26,3.73,7.77
7,80,3.27,0.00,2.78,1.98,5.41,3.20,3.70,6.40,9.61
8,90,0.00,0.00,0.00,14.29,0.00,0.00,0.00,8.33,13.33


### 2.4 Canal × système d'exploitation avec sous-totaux (`ROLLUP`)

In [19]:
sql("""
    SELECT coalesce(source, 'TOUS')      AS source,
           coalesce(device_os, 'TOUS')   AS device_os,
           count(*)                      AS demandes,
           round(100 * avg(fraud_bool), 3) AS taux_fraude_pct
    FROM applications_clean
    GROUP BY ROLLUP (source, device_os)
    ORDER BY source = 'TOUS', source, device_os = 'TOUS', taux_fraude_pct DESC
""")

,source,device_os,demandes,taux_fraude_pct
0,INTERNET,windows,263005,2.463
1,INTERNET,macintosh,53599,1.384
2,INTERNET,x11,7200,1.125
3,INTERNET,other,337312,0.566
4,INTERNET,linux,331836,0.514
5,INTERNET,TOUS,992952,1.099
6,TELEAPP,windows,501,5.788
7,TELEAPP,macintosh,227,4.405
8,TELEAPP,other,5416,1.182
9,TELEAPP,linux,876,1.027


### 2.5 Profil des demandes frauduleuses et légitimes
Médianes pour les variables continues, proportions pour les indicateurs binaires.

In [20]:
profil = sql("""
    SELECT
        CASE fraud_bool WHEN 1 THEN 'fraude' ELSE 'légitime' END    AS classe,
        count(*)                                                    AS demandes,
        median(credit_risk_score)                                   AS credit_risk_score_med,
        round(median(name_email_similarity), 3)                     AS name_email_similarity_med,
        median(proposed_credit_limit)                               AS proposed_credit_limit_med,
        round(median(session_length_in_minutes), 2)                 AS session_length_med,
        median(customer_age)                                        AS customer_age_med,
        round(100 * avg(email_is_free), 1)                          AS pct_email_gratuit,
        round(100 * avg(phone_home_valid), 1)                       AS pct_tel_fixe_valide,
        round(100 * avg(foreign_request), 1)                        AS pct_demande_etrangere,
        round(100 * avg(keep_alive_session), 1)                     AS pct_rester_connecte,
        round(100 * avg(has_other_cards), 1)                        AS pct_autres_cartes,
        round(100 * avg((source = 'TELEAPP')::INT), 1)                     AS pct_teleapp
    FROM applications_clean
    GROUP BY fraud_bool
    ORDER BY fraud_bool DESC
""")
profil.set_index("classe").T

classe,fraude,légitime
demandes,11029.000,988971.000
credit_risk_score_med,178.000,122.000
name_email_similarity_med,0.292,0.494
proposed_credit_limit_med,500.000,200.000
session_length_med,5.130,5.120
customer_age_med,40.000,30.000
pct_email_gratuit,66.100,52.800
pct_tel_fixe_valide,25.300,41.900
pct_demande_etrangere,5.000,2.500
pct_rester_connecte,34.200,58.000


## 3. Fonctions de fenêtre

### 3.1 Évolution mensuelle
Sur l'agrégat mensuel :
- `LAG` : variation par rapport au mois précédent
- `SUM(...) OVER (ORDER BY month)` : taux de fraude cumulé
- `AVG(...) OVER (ROWS BETWEEN 2 PRECEDING AND CURRENT ROW)` : moyenne mobile sur 3 mois
- `SUM(...) OVER ()` : part du mois dans le volume total

In [21]:
sql("""
    WITH mensuel AS (
        SELECT month, count(*) AS demandes, sum(fraud_bool)::BIGINT AS fraudes, avg(fraud_bool) AS taux
        FROM applications_clean
        GROUP BY month
    )
    SELECT
        month,
        demandes,
        fraudes,
        round(100 * taux, 3)                                                       AS taux_fraude_pct,
        round(100 * (taux - lag(taux) OVER w), 3)                                  AS variation_pts,
        round(100 * sum(fraudes) OVER w / sum(demandes) OVER w, 3)                 AS taux_cumule_pct,
        round(100 * avg(taux) OVER (w ROWS BETWEEN 2 PRECEDING AND CURRENT ROW), 3) AS moyenne_mobile_3m_pct,
        round(100 * demandes / sum(demandes) OVER (), 2)                           AS part_volume_pct
    FROM mensuel
    WINDOW w AS (ORDER BY month)
    ORDER BY month
""")

,month,demandes,fraudes,taux_fraude_pct,variation_pts,taux_cumule_pct,moyenne_mobile_3m_pct,part_volume_pct
0,0,132440,1500,1.133,NaN,1.133,1.133,13.24
1,1,127620,1198,0.939,-0.194,1.037,1.036,12.76
2,2,136979,1198,0.875,-0.064,0.981,0.982,13.70
3,3,150936,1392,0.922,0.048,0.965,0.912,15.09
4,4,127691,1452,1.137,0.215,0.998,0.978,12.77
5,5,119323,1411,1.183,0.045,1.025,1.081,11.93
6,6,108168,1450,1.341,0.158,1.063,1.220,10.82
7,7,96843,1428,1.475,0.134,1.103,1.333,9.68


### 3.2 Classement des modalités et lift
On reprend l'agrégat de la section 2.2 et on ajoute :
- `RANK() OVER (PARTITION BY variable ...)` : rang de la modalité au sein de sa variable
- `lift` : taux de la modalité / taux global
- `part_fraudes_pct` : part des fraudes de la variable captée par la modalité

In [22]:
sql(f"""
    WITH par_modalite AS ({par_modalite}),
    global AS (SELECT avg(fraud_bool) AS taux_global FROM applications_clean)
    SELECT
        variable,
        modalite,
        demandes,
        taux_fraude_pct,
        rank() OVER (PARTITION BY variable ORDER BY taux_fraude_pct DESC)             AS rang,
        round(taux_fraude_pct / (100 * taux_global), 2)                              AS lift,
        round(100 * fraudes / sum(fraudes) OVER (PARTITION BY variable), 1)          AS part_fraudes_pct,
        round(100 * demandes / sum(demandes) OVER (PARTITION BY variable), 1)        AS part_demandes_pct
    FROM par_modalite, global
    QUALIFY rang <= 3
    ORDER BY variable, rang
""")

,variable,modalite,demandes,taux_fraude_pct,rang,lift,part_fraudes_pct,part_demandes_pct
0,device_os,windows,263506,2.469,1,2.24,59.0,26.4
1,device_os,macintosh,53826,1.397,2,1.27,6.8,5.4
2,device_os,x11,7228,1.121,3,1.02,0.7,0.7
3,employment_status,CC,37758,2.468,1,2.24,8.5,3.8
4,employment_status,CG,453,1.545,2,1.40,0.1,0.0
5,employment_status,CA,730252,1.219,3,1.11,80.7,73.0
6,housing_status,BA,169675,3.747,1,3.40,57.6,17.0
7,housing_status,BD,26161,0.864,2,0.78,2.0,2.6
8,housing_status,BC,372143,0.615,3,0.56,20.7,37.2
9,payment_type,AC,252071,1.670,1,1.51,38.2,25.2


### 3.3 Déciles de `credit_risk_score` et courbe de gain (`NTILE`)
`NTILE(10)` découpe les demandes en 10 groupes de même taille selon le score.
Le cumul des fraudes en partant du décile le plus risqué donne la part des fraudes captées si l'on contrôle les X % de demandes les plus risquées.

In [23]:
sql("""
    WITH deciles AS (
        SELECT fraud_bool, credit_risk_score,
               ntile(10) OVER (ORDER BY credit_risk_score DESC) AS decile   -- 1 = scores les plus élevés
        FROM applications_clean
    ),
    agg AS (
        SELECT decile,
               min(credit_risk_score) AS score_min,
               max(credit_risk_score) AS score_max,
               count(*)               AS demandes,
               sum(fraud_bool)::BIGINT        AS fraudes
        FROM deciles
        GROUP BY decile
    )
    SELECT
        decile,
        score_min,
        score_max,
        demandes,
        fraudes,
        round(100 * fraudes / demandes, 2)                                                   AS taux_fraude_pct,
        round(100 * sum(fraudes) OVER (ORDER BY decile) / sum(fraudes) OVER (), 1)          AS fraudes_captees_cumul_pct,
        round((fraudes / demandes) / (sum(fraudes) OVER () / sum(demandes) OVER ()), 2)     AS lift
    FROM agg
    ORDER BY decile
""")

,decile,score_min,score_max,demandes,fraudes,taux_fraude_pct,fraudes_captees_cumul_pct,lift
0,1,226,389,100000,3360,3.36,30.5,3.05
1,2,192,226,100000,1542,1.54,44.4,1.40
2,3,165,192,100000,1261,1.26,55.9,1.14
3,4,141,165,100000,996,1.00,64.9,0.90
4,5,122,141,100000,688,0.69,71.1,0.62
5,6,107,122,100000,614,0.61,76.7,0.56
6,7,92,107,100000,632,0.63,82.4,0.57
7,8,73,92,100000,713,0.71,88.9,0.65
8,9,48,73,100000,682,0.68,95.1,0.62
9,10,-170,48,100000,541,0.54,100.0,0.49


### 3.4 Variables calculées par ligne
On construit une table temporaire `applications_features` avec des variables relatives au contexte de chaque demande :

| Variable | Fenêtre | Sens |
|---|---|---|
| `credit_risk_pct_rank_mois` | `PERCENT_RANK()` par mois | position du score parmi les demandes du même mois (0 à 1) |
| `velocity_6h_zscore_mois` | `AVG` / `STDDEV` par mois | écart à la normale du mois, en écarts-types |
| `credit_limit_ratio_logement` | `AVG` par `housing_status` | limite proposée / limite moyenne du même statut de logement |
| `session_ecart_mediane_os` | `MEDIAN` par `device_os` | durée de session moins la médiane du même OS |
| `nb_demandes_meme_profil_mois` | `COUNT` par (mois, âge, revenu, OS) | taille du groupe de demandes au profil identique dans le mois |
| `taux_fraude_hist_paiement` | `SUM ... ROWS BETWEEN UNBOUNDED PRECEDING AND 1 PRECEDING` | taux de fraude du même `payment_type` sur les **mois précédents** uniquement (pas de fuite d'information) |
| `nb_manquants` | aucune (somme de colonnes) | nombre de variables non renseignées |

In [24]:
con.sql("""
    CREATE OR REPLACE TEMP TABLE applications_features AS
    WITH hist_paiement AS (
        -- taux de fraude historique par type de paiement, calculé sur les mois strictement antérieurs
        SELECT payment_type, month,
               sum(sum(fraud_bool)) OVER w / sum(count(*)) OVER w AS taux_fraude_hist_paiement
        FROM applications_clean
        GROUP BY payment_type, month
        WINDOW w AS (PARTITION BY payment_type ORDER BY month
                     ROWS BETWEEN UNBOUNDED PRECEDING AND 1 PRECEDING)
    )
    SELECT
        a.*,
        percent_rank() OVER (PARTITION BY month ORDER BY credit_risk_score)              AS credit_risk_pct_rank_mois,
        (velocity_6h - avg(velocity_6h) OVER (PARTITION BY month))
            / stddev(velocity_6h) OVER (PARTITION BY month)                              AS velocity_6h_zscore_mois,
        proposed_credit_limit / avg(proposed_credit_limit) OVER (PARTITION BY housing_status)
                                                                                         AS credit_limit_ratio_logement,
        session_length_in_minutes - median(session_length_in_minutes) OVER (PARTITION BY device_os)
                                                                                         AS session_ecart_mediane_os,
        count(*) OVER (PARTITION BY month, customer_age, income, device_os)              AS nb_demandes_meme_profil_mois,
        h.taux_fraude_hist_paiement,
        prev_address_months_count_manquant + current_address_months_count_manquant
            + bank_months_count_manquant + session_length_in_minutes_manquant
            + device_distinct_emails_8w_manquant + intended_balcon_amount_manquant       AS nb_manquants
    FROM applications_clean a
    LEFT JOIN hist_paiement h USING (payment_type, month)
""")

sql("""
    SELECT application_id, month, payment_type, fraud_bool,
           round(credit_risk_pct_rank_mois, 3)   AS credit_risk_pct_rank_mois,
           round(velocity_6h_zscore_mois, 3)     AS velocity_6h_zscore_mois,
           round(credit_limit_ratio_logement, 3) AS credit_limit_ratio_logement,
           round(session_ecart_mediane_os, 2)    AS session_ecart_mediane_os,
           nb_demandes_meme_profil_mois,
           round(taux_fraude_hist_paiement, 4)   AS taux_fraude_hist_paiement,
           nb_manquants
    FROM applications_features
    ORDER BY application_id
    LIMIT 10
""")

,application_id,month,payment_type,fraud_bool,credit_risk_pct_rank_mois,velocity_6h_zscore_mois,credit_limit_ratio_logement,session_ecart_mediane_os,nb_demandes_meme_profil_mois,taux_fraude_hist_paiement,nb_manquants
0,1,0,AA,0,0.731,1.877,3.232,10.89,557,NaN,1
1,2,0,AD,0,0.698,0.593,3.232,-1.56,2326,NaN,2
2,3,0,AB,0,0.415,-0.984,0.431,17.57,1157,NaN,1
3,4,0,AB,0,0.421,2.321,0.431,9.88,1475,NaN,1
4,5,0,AA,0,0.426,0.055,0.431,-1.18,2900,NaN,1
5,6,0,AD,0,0.617,1.367,0.469,1.65,1475,NaN,2
6,7,0,AB,0,0.308,1.422,0.431,22.15,20,NaN,1
7,8,0,AB,0,0.731,-0.809,0.469,6.31,1809,NaN,2
8,9,0,AB,0,0.098,-0.152,0.431,0.41,755,NaN,1
9,10,0,AD,0,0.860,0.037,2.798,-0.82,2326,NaN,2


Contrôle du taux historique : pour le mois 0 il n'y a pas d'historique (`NULL`), puis le taux s'appuie sur un nombre croissant de mois.

In [25]:
sql("""
    SELECT payment_type, month, round(100 * any_value(taux_fraude_hist_paiement), 3) AS taux_hist_pct
    FROM applications_features
    WHERE payment_type IN ('AA', 'AC')
    GROUP BY ALL
    ORDER BY payment_type, month
""")

,payment_type,month,taux_hist_pct
0,AA,0,NaN
1,AA,1,0.849
2,AA,2,0.635
3,AA,3,0.549
4,AA,4,0.520
5,AA,5,0.511
6,AA,6,0.502
7,AA,7,0.512
8,AC,0,NaN
9,AC,1,1.918


### 3.5 Pouvoir discriminant des nouvelles variables
Taux de fraude par quintile de chaque variable calculée (`NTILE(5)` sur la variable, puis `PIVOT`).

In [26]:
sql("""
    WITH long AS (
        UNPIVOT (SELECT fraud_bool, credit_risk_pct_rank_mois, velocity_6h_zscore_mois,
                        credit_limit_ratio_logement, session_ecart_mediane_os,
                        nb_demandes_meme_profil_mois::DOUBLE AS nb_demandes_meme_profil_mois,
                        nb_manquants::DOUBLE AS nb_manquants
                 FROM applications_features)
        ON COLUMNS(* EXCLUDE (fraud_bool)) INTO NAME variable VALUE valeur
    ),
    quintiles AS (
        SELECT variable, fraud_bool,
               'Q' || ntile(5) OVER (PARTITION BY variable ORDER BY valeur) AS quintile
        FROM long
    )
    PIVOT quintiles
    ON quintile
    USING round(100 * avg(fraud_bool), 2)
    GROUP BY variable
    ORDER BY variable
""")

,variable,Q1,Q2,Q3,Q4,Q5
0,credit_limit_ratio_logement,1.28,0.52,0.39,1.45,1.88
1,credit_risk_pct_rank_mois,0.60,0.66,0.71,1.11,2.43
2,nb_demandes_meme_profil_mois,1.51,0.97,0.87,1.08,1.07
3,nb_manquants,0.38,0.56,1.25,1.29,2.04
4,session_ecart_mediane_os,1.02,1.14,1.17,1.11,1.07
5,velocity_6h_zscore_mois,1.32,1.13,1.06,1.10,0.90


### 3.6 Demandes atypiques par mois (`QUALIFY`)
Les 3 demandes les plus éloignées de la vitesse moyenne du mois, sans sous-requête grâce à `QUALIFY`.

In [27]:
sql("""
    SELECT month, application_id, round(velocity_6h, 1) AS velocity_6h,
           round(velocity_6h_zscore_mois, 2) AS zscore, fraud_bool,
           row_number() OVER (PARTITION BY month ORDER BY velocity_6h_zscore_mois DESC) AS rang
    FROM applications_features
    WHERE velocity_6h_zscore_mois IS NOT NULL
    QUALIFY rang <= 3
    ORDER BY month, rang
""")

,month,application_id,velocity_6h,zscore,fraud_bool,rang
0,0,28355,16715.6,3.08,0,1
1,0,7175,16606.6,3.04,0,2
2,0,5221,16451.8,2.99,0,3
3,1,166026,16665.4,3.02,0,1
4,1,164660,16640.7,3.01,0,2
5,1,247107,16579.1,2.99,0,3
6,2,364887,16595.7,3.24,0,1
7,2,383929,16514.4,3.22,0,2
8,2,269127,16435.4,3.19,0,3
9,3,547516,16605.5,3.80,0,1


## Synthèse

**Qualité**
- 1 000 000 demandes, aucun doublon, aucun `NULL`. En revanche, les manquants sont **codés** (`-1` ou valeur négative) : 74 % pour `intended_balcon_amount`, 71 % pour `prev_address_months_count`, 25 % pour `bank_months_count`.
- Ces manquants sont informatifs : une adresse précédente non renseignée correspond à 1,42 % de fraude contre 0,31 % sinon. D'où les indicateurs `*_manquant`.
- `device_fraud_count` est constante (supprimée) ; 44 valeurs négatives de `velocity_6h` sont mises à `NULL`.
- Les mois 2 et 4 sont répartis sur deux fichiers source : il faut toujours travailler sur la table fusionnée.

**Agrégations**
- Taux de fraude global : 1,10 %. Il augmente en fin de période (0,88 % au mois 2, 1,48 % au mois 7) : l'échantillon test (mois 6-7, 1,40 %) est plus fraudé que le train (1,03 %). Il faudra en tenir compte lors de l'évaluation.
- Segments à risque : `housing_status = BA` (3,75 %, 58 % des fraudes pour 17 % des demandes), `device_os = windows` (2,47 %, 59 % des fraudes), `employment_status = CC` (2,47 %).
- La fraude augmente avec l'âge et le revenu : au-delà de 60 ans avec un revenu de 0,9, le taux dépasse 5 % (effectifs faibles pour 80 et 90 ans).
- Les fraudeurs ont un nom moins proche de leur email, un score de risque plus élevé, une limite de crédit proposée plus haute, et activent moins l'option « rester connecté ».

**Fonctions de fenêtre**
- Les 10 % de demandes au `credit_risk_score` le plus élevé captent 30,5 % des fraudes (lift 3,05). Ce pouvoir discriminant confirme le point « fuite possible » noté dans le dictionnaire des variables.
- Parmi les variables calculées, `credit_risk_pct_rank_mois` (2,43 % de fraude dans le quintile haut contre 0,60 % dans le bas) et `nb_manquants` (2,04 % contre 0,34 %) sont les plus discriminantes ; `session_ecart_mediane_os` n'apporte presque rien.
- `taux_fraude_hist_paiement` n'utilise que les mois passés : il peut servir de variable de modèle sans fuite d'information.

In [28]:
con.close()